<a href="https://colab.research.google.com/github/kaustav02github/Toxicity_Classifier/blob/main/Toxicity_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files
files.upload()


KeyboardInterrupt: 

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle


# Downloading **Datasets**

In [ ]:
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge


In [ ]:
#unzip file
!unzip -q jigsaw-toxic-comment-classification-challenge.zip -d jigsaw


In [ ]:
!unzip -q jigsaw/train.csv.zip -d jigsaw
!unzip -q jigsaw/test.csv.zip -d jigsaw

In [ ]:
df=pd.read_csv('jigsaw/train.csv')

In [ ]:
df.tail()

In [ ]:
df.iloc[:2]

In [ ]:
df.iloc[43].comment_text

## This line of code *bellow* selects a subset of the DataFrame df. It keeps all rows but only includes the columns from the third column onwards. It's like creating a smaller table from your original table by choosing specific columns.

In [ ]:
df[df.columns[2:]]

In [ ]:
df[df.columns[2:]].iloc[43]

# PreProcessing Data :

> Add blockquote



In [ ]:
from tensorflow.keras.layers import TextVectorization

#this line over here will convert certain words to integer
#like (I hate u ->where I=23,hate =44,u-85) this is an eg

In [ ]:
X=df['comment_text']
y=df[df.columns[2:]].values
y

In [ ]:
Max_featues=200000 #number of words in vocab

In [ ]:
vectorrizer=TextVectorization(max_tokens=Max_featues,output_sequence_length=1800,output_mode='int')

In [ ]:
vectorrizer.adapt(X.values)
 #assigning integer values to each word and ignoring the punctuations

In [ ]:
vectorrizer("Hello how U doing")[:5]
#check the numpy array bellow each words maps to a integer

In [ ]:
vectorized_text=vectorrizer(X.values)
 #vectorizeed text will contain all equivalent integer array of the sentences

In [ ]:
dataset=tf.data.Dataset.from_tensor_slices((vectorized_text,y))#combining the dataset
dataset=dataset.cache()
dataset=dataset.shuffle(160000)
dataset=dataset.batch(16)
dataset=dataset.prefetch(8)

In [ ]:
train = dataset.take(int(len(dataset)*.7))
val = dataset.skip(int(len(dataset)*.7)).take(int(len(dataset)*.2))
test = dataset.skip(int(len(dataset)*.9)).take(int(len(dataset)*.1))

In [ ]:
train_generator=train.as_numpy_iterator()
train_generator.next()

# Traing our model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional, Dense, Embedding

In [ ]:
model = Sequential()
# Create the embedding layer
model.add(Embedding(Max_featues+1, 32))
# Bidirectional LSTM Layer
model.add(Bidirectional(LSTM(32, activation='tanh')))
# Feature extractor Fully connected layers
model.add(Dense(128, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dense(128, activation='relu'))
# Final layer
model.add(Dense(6, activation='sigmoid'))

In [ ]:
model.compile(loss='BinaryCrossentropy', optimizer='Adam',metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
history=model.fit(train,epochs=1,validation_data=val)

# Model **Predictions**

In [ ]:
input_text=vectorrizer("You freaking suck at it,I will kill u")

In [ ]:
input_text

In [ ]:
res=model.predict(np.expand_dims(input_text,0))

In [ ]:
df.columns[2:]

In [ ]:
(res >0.5).astype(int)

In [ ]:
batch_x,batch_y=test.as_numpy_iterator().next()

In [ ]:
batch_x

In [ ]:
# prompt: predict for batch_x
pred=model.predict(batch_x)
(pred>0.5).astype(int)

In [ ]:
batch_y

# Evaluate Model

In [ ]:
from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy
pre = Precision()
re = Recall()
acc = CategoricalAccuracy()

In [ ]:
for batch in test.as_numpy_iterator():
    # Unpack the batch
    X_true, y_true = batch
    # Make a prediction
    yhat = model.predict(X_true)

    # Flatten the predictions
    y_true = y_true.flatten()
    yhat = yhat.flatten()

    pre.update_state(y_true, yhat)
    re.update_state(y_true, yhat)
    acc.update_state(y_true, yhat)

In [ ]:
print(f'Precision: {pre.result().numpy()}, Recall:{re.result().numpy()}, Accuracy:{acc.result().numpy()}')


NameError: name 'pre' is not defined

# Gradio App interface